# Diseño y preparación de la base de datos y Data Mart

Este notebook documenta el proceso para pasar del dataset procesado a una base de datos relacional y a un Data Mart dimensional. Se alinea con las indicaciones del Proyecto Integrador: documentar la limpieza y el ETL, y construir un Data Mart con tabla de hechos, dimensiones, llaves y medidas.

**Motor previsto:** Microsoft SQL Server, sujeto a confirmar en el equipo donde se ejecutará el proyecto.

**Fuente de carga:** `data/processed/online_shoppers_intention_clean.csv`. El archivo original en `data/raw/` se conserva sin cambios.

## 1. Estado y objetivo

El dataset procesado disponible contiene 12,205 sesiones y 18 variables. En la revisión local se observaron cero duplicados exactos y cero valores faltantes. La variable `Month` está estandarizada con abreviaturas; por ejemplo, `June` se transformó en `Jun`.

El objetivo de esta etapa es almacenar las sesiones y sus atributos para consultar el comportamiento de compra. La base tendrá una zona de staging para recibir el CSV y un esquema estrella para el Data Mart.

In [3]:
from pathlib import Path
import pandas as pd

ruta_csv = Path('../data/processed/online_shoppers_intention_clean.csv')
datos = pd.read_csv(ruta_csv)

print(f'Filas: {datos.shape[0]:,}')
print(f'Columnas: {datos.shape[1]}')
print(f'Duplicados exactos: {datos.duplicated().sum():,}')
print(f'Valores faltantes: {datos.isna().sum().sum():,}')
print('Columnas:', datos.columns.tolist())

Filas: 12,205
Columnas: 18
Duplicados exactos: 0
Valores faltantes: 0
Columnas: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']


## 2. Grano y reglas del modelo

**Grano de `FactSesion`: una fila por cada sesión del dataset procesado.** El archivo fuente no incluye un identificador único de sesión, por lo que SQL Server generará una llave sustituta (`SesionKey`) para identificar cada fila dentro del Data Mart. Esa llave no representa un identificador original del sitio web.

Las variables categóricas se separan en dimensiones para facilitar filtros y agrupaciones. Algunas se almacenan como números en el CSV, pero representan códigos, no cantidades. Las variables de páginas, duración y tasas se conservan como medidas. `Revenue` se conserva como indicador de compra y variable objetivo para clasificación; `Weekend` se conserva como indicador de sesión.

## 3. Mapeo de variables a tablas

| Tabla | Variables | Uso |
|---|---|---|
| `StgSesiones` | Las 18 columnas del CSV más `StagingKey` generado | Recibir la carga procesada y conservar una fila fuente por sesión. |
| `DimMes` | `Month` | Filtrar y agrupar por mes. |
| `DimTipoVisitante` | `VisitorType` | Analizar visitantes nuevos, recurrentes y otros. |
| `DimSistemaOperativo` | `OperatingSystems` | Analizar el código de sistema operativo. |
| `DimNavegador` | `Browser` | Analizar el código de navegador. |
| `DimRegion` | `Region` | Analizar el código de región. |
| `DimTipoTrafico` | `TrafficType` | Analizar el código de tipo de tráfico. |
| `FactSesion` | Llaves a dimensiones; cantidades de páginas; duraciones; tasas; `PageValues`; `SpecialDay`; `Weekend`; `Revenue` | Mantener una fila por sesión y sus medidas/indicadores. |

Las dimensiones tendrán llaves sustitutas enteras como PK. `FactSesion` guardará esas llaves como FK. Para evitar duplicar una sesión al reejecutar la carga, el hecho conservará la llave de origen de staging con restricción única.

## 4. Orden del proceso

1. Crear la base `ECBD_Compradores` en SQL Server.
2. Crear `StgSesiones` con las 18 variables del CSV y una llave técnica generada por SQL Server.
3. Importar el CSV procesado a staging, sin modificar el archivo original.
4. Crear las dimensiones y `FactSesion` con PK, FK, tipos de datos y medidas.
5. Poblar cada dimensión con los valores distintos presentes en staging.
6. Poblar `FactSesion` relacionando cada sesión con sus llaves dimensionales.
7. Validar conteos, valores nulos y relaciones entre staging y Data Mart.
8. Guardar los scripts SQL y documentar el resultado del ETL.

### Paso 1: crear la base de datos

Ejecutar en una consulta de SQL Server Management Studio conectada al servidor. `USE` selecciona la base para las instrucciones posteriores.

```sql
IF DB_ID(N'ECBD_Compradores') IS NULL
BEGIN
    CREATE DATABASE ECBD_Compradores;
END;
GO

USE ECBD_Compradores;
GO

SELECT DB_NAME() AS BaseDeDatosActual;
```

La creación de la base de datos, la tabla de staging, las dimensiones y la tabla de hechos se encuentra implementada en el script `sql/01_crear_base_y_datamart.sql`.

La carga del dataset procesado y la construcción del Data Mart se encuentran implementadas en `sql/02_cargar_dataset.sql`. Ambos scripts deberán ejecutarse en SQL Server Management Studio para comprobar su funcionamiento.

## 5. Validaciones de carga previstas

Al terminar el ETL, comparar el número de filas de staging y de `FactSesion`. Como el dataset procesado tiene una fila por sesión y las dimensiones se construyen con valores distintos, la carga del hecho debe conservar las 12,205 sesiones. También se comprobará que no haya llaves foráneas sin correspondencia y que las dimensiones no tengan duplicados en sus atributos naturales.

Consultas de control incluidas en el script `sql/02_cargar_dataset.sql` para revisar los registros cargados:

```sql
SELECT COUNT(*) AS FilasStaging FROM dbo.StgSesiones;
SELECT COUNT(*) AS FilasHechos FROM dbo.FactSesion;
SELECT TOP (5) * FROM dbo.FactSesion;
```

## 6. Estado de esta etapa

- Dataset procesado disponible: **sí**.
- Diseño conceptual y grano del hecho documentados: **sí**.
- Creación de la base, staging, dimensiones y tabla de hechos: **implementada en scripts SQL; pendiente de ejecución**.
- Carga del dataset y proceso ETL: **implementados en scripts SQL; pendientes de ejecución y validación**.
- Diagrama del Data Mart: **elaborado**.
- Validación de cantidades de registros y relaciones: **pendiente de comprobar en SQL Server**.

Este notebook documenta el diseño y permite revisar el dataset procesado con Python. La ejecución de los scripts SQL se realiza en SQL Server Management Studio.